# Notebook 01 — Initial Data Exploration

**Phase 1 learning checkpoint.** Before we touch any model, we have to *meet* the data.

## What you will do here

1. Load the two raw datasets that `src/data_loader.py` just downloaded.
2. Look at their **shape**, **schema**, and **a few sample rows**.
3. Run a missingness audit — which columns are sparse?
4. Get a feel for the **target** (`SalePrice`) — is it skewed?
5. Sanity-check the Zillow time series — does it really cover the years it claims?

## Reading order

Pair this notebook with [`docs/01_data_science_fundamentals.md`](../docs/01_data_science_fundamentals.md) (the *why*) and [`docs/02_understanding_the_data.md`](../docs/02_understanding_the_data.md) (the *what*).

> ⚠️ **Beginner pitfall warning.** Don't be tempted to skip this notebook and jump straight to modeling. "Understanding the data" *is* the job for the first 30–50% of any real project. Skipping it produces models that look great on paper and fail in production.

## 0. Setup

We import our own `data_loader` module so notebook and production code use the **same** load logic — no copy-paste drift.

In [ ]:
# This 'magic' makes edits to src/*.py auto-reload without restarting the kernel.
# Saves you a lot of frustration during development.
%load_ext autoreload
%autoreload 2

# Make project root importable so `from src.data_loader import ...` works
# from inside notebooks/.
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent  # notebooks/ -> project root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Show all columns; pandas truncates wide DataFrames by default and Ames is wide.
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

from src.data_loader import load_ames, load_zillow_zhvi
print("Setup complete.")

## 1. Ames Housing — first look

We expect ~2,930 rows and ~82 columns. Each row is one home sale in Ames, IA between 2006 and 2010. The target variable we'll predict later is `SalePrice`.

In [ ]:
ames = load_ames()
print(f"Shape: {ames.shape}")
ames.head()

### 1a. Column inventory

How many numeric vs. categorical columns? This drives every preprocessing decision in Phase 2.

In [ ]:
dtypes_summary = ames.dtypes.value_counts()
print("Dtype breakdown:")
print(dtypes_summary)
print(f"\nTotal numeric:    {ames.select_dtypes(include=np.number).shape[1]}")
print(f"Total categorical: {ames.select_dtypes(include='object').shape[1]}")

### 1b. Missingness audit

A column with 80% missing values is usually unusable. A column with 5% missing is often fine after imputation. **Knowing which is which is the first decision of Phase 2.**

> 📘 **Concept — "NA" can be meaningful.** In the Ames data, many `NA`s do **not** mean "unknown". They mean "this feature does not exist for this home" (e.g. `PoolQC = NA` means "no pool"). The data dictionary in `data/raw/ames_data_dictionary.txt` tells you which `NA`s are structural vs. truly missing — read it. We'll handle this carefully in Phase 2.

In [ ]:
missing = ames.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(ames) * 100).round(2)
missingness = pd.DataFrame({"n_missing": missing, "pct_missing": missing_pct})
missingness = missingness[missingness["n_missing"] > 0]
print(f"{len(missingness)} columns have at least one missing value:")
missingness.head(20)

### 1c. The target — `SalePrice`

Distribution shape matters. If `SalePrice` is right-skewed (long tail of expensive homes), some models do better on `log(SalePrice)` than on raw price. We'll explore that in Phase 3, but a quick histogram now plants the seed.

In [ ]:
print(ames["SalePrice"].describe().round(0))
print(f"\nSkewness: {ames['SalePrice'].skew():.2f}   (>1 means notably right-skewed)")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(ames["SalePrice"], bins=40)
axes[0].set_title("SalePrice — raw")
axes[0].set_xlabel("USD")
axes[1].hist(np.log1p(ames["SalePrice"]), bins=40)
axes[1].set_title("SalePrice — log1p")
axes[1].set_xlabel("log(USD + 1)")
plt.tight_layout()
plt.show()

## 2. Zillow ZHVI — first look

This is structurally very different from Ames: one row per US ZIP code, and one column for **every** month from 2000 to today. Look at the column types carefully — most should be dates-as-strings.

In [ ]:
zhvi = load_zillow_zhvi()
print(f"Shape: {zhvi.shape}")
zhvi.iloc[:5, :8]

### 2a. Identify ID columns vs. value columns

The first handful of columns describe the ZIP (region name, state, metro). The rest are monthly value columns. We separate them so we can talk about each.

In [ ]:
# Heuristic: a column whose name looks like a date (YYYY-MM-DD) is a value
# column. Everything else is metadata.
import re
date_col_pattern = re.compile(r"^\d{4}-\d{2}-\d{2}$")
date_cols = [c for c in zhvi.columns if date_col_pattern.match(c)]
id_cols = [c for c in zhvi.columns if c not in date_cols]

print(f"ID/metadata columns ({len(id_cols)}): {id_cols}")
print(f"Date columns: {len(date_cols)}  (from {date_cols[0]} to {date_cols[-1]})")

### 2b. Geographic coverage

How many ZIPs and which states? This tells us whether we can do a national heatmap or should restrict to one metro for now.

In [ ]:
print(f"Unique ZIPs: {zhvi['RegionName'].nunique():,}")
print(f"States covered: {zhvi['State'].nunique()}")
print("\nTop 5 states by number of ZIPs:")
print(zhvi['State'].value_counts().head())

### 2c. Sanity-check a known ZIP

Pick an Ames, Iowa ZIP (50010) and plot its price trajectory. This is also our first time-series chart, and it primes the Phase 5 forecasting work.

In [ ]:
ames_zip_row = zhvi[zhvi["RegionName"] == 50010]
if not ames_zip_row.empty:
    series = ames_zip_row[date_cols].iloc[0]
    series.index = pd.to_datetime(series.index)
    series = series.dropna()
    series.plot(figsize=(10, 4), title="ZHVI for ZIP 50010 (Ames, IA)")
    plt.ylabel("Typical home value (USD)")
    plt.xlabel("")
    plt.show()
    print(f"Series spans {series.index.min().date()} to {series.index.max().date()},"
          f" with {len(series)} non-null monthly observations.")
else:
    print("ZIP 50010 not in dataset — Zillow's coverage may have changed. Pick another.")

## 3. Wrap-up — what did we learn?

Write your own answers below before moving on:

1. **Ames dataset shape and target:** how many rows? How many features? What is the range of `SalePrice`?
2. **Most-missing columns:** name the top 3 — are they candidates for *imputation* or *dropping*? (Hint: check the data dictionary for whether `NA` is structural.)
3. **Target skew:** is the raw target skewed enough that you'd consider a log-transform?
4. **Zillow coverage:** does the dataset go far enough back / forward to give us useful trend signal?
5. **Joining the two datasets:** Ames is by individual home; Zillow is by ZIP. What's the granularity mismatch and what does that mean for how we'll combine them later?

Once you can answer those without re-reading the cells above, you're ready for **Phase 2: Data Cleaning & EDA**.